### Test delta lake installation

### Spark session 

In [ ]:
import os
from pathlib import Path
from pyspark.sql import SparkSession

# Windows: Spark needs Hadoop's winutils.exe/hadoop.dll (in C:\hadoop\bin).
# This MUST run before the first SparkSession is created: the Spark JVM reads
# HADOOP_HOME only when it starts, so setting it afterwards has no effect.
os.environ["HADOOP_HOME"] = r"C:\hadoop"
hadoop_bin = os.path.join(os.environ["HADOOP_HOME"], "bin")
if hadoop_bin not in os.environ["PATH"]:
    os.environ["PATH"] += os.pathsep + hadoop_bin

### Install Apache Spark and Delta Lake
-- pip install pyspark delta-spark


###  Step 2: Configure Delta Lake with Spark

The Spark session is set up with two critical configurations. The first enables 

**Delta Lake** SQL extensions and the second defines Delta Lake as the default 

**Catalog**, ensuring that your Delta-format data is correctly handled.

In [ ]:
from delta import configure_spark_with_delta_pip
from pyspark import SparkContext
from pyspark.sql import SparkSession

# The Delta JARs are only loaded when Spark's JVM starts. If this kernel already
# started Spark without them, getOrCreate() would hand back that old session and
# writes fail with DATA_SOURCE_NOT_FOUND. Only a kernel restart fixes that.
sc = SparkContext._active_spark_context
if sc is not None and "delta" not in sc.getConf().get("spark.jars.packages", ""):
    raise RuntimeError(
        "Spark was already started in this kernel without Delta Lake. "
        "Restart the kernel (Restart button in the notebook toolbar), then run all cells from the top."
    )

# configure_spark_with_delta_pip adds the Delta Lake JARs (spark.jars.packages).
builder = (
    SparkSession.builder
    .appName("DeltaLakePractice")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

# Check if Spark is working
print("Spark Session Created Successfully!")


### Delta Lake Basics

In [ ]:
# Sample DataFrame creation
data = [("Alice", 34), ("Bob", 45), ("Cathy", 29)]
columns = ["name", "age"]
df = spark.createDataFrame(data, columns)

# Save the DataFrame as a Delta table (overwrite mode replaces any existing data).
path = str(Path(r"C:\Projects\deltalake\src\deltalake\tables"))
df.write.format("delta").mode("overwrite").save(path)

# Read it back to confirm
spark.read.format("delta").load(path).show()

from pathlib import Path
from tempfile import TemporaryDirectory
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

builder = (
    SparkSession.builder
    .master("local[1]")
    .appName("delta-provider-smoke-test")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
try:
    with TemporaryDirectory(prefix="delta-smoke-") as temp_dir:
        path = str(Path(temp_dir) / "table")
        spark.createDataFrame([("Alice", 34)], ["name", "age"]).write.format("delta").save(path)
        result = spark.read.format("delta").load(path).collect()
        print(f"Delta round-trip succeeded: {result}")
finally:
    spark.stop()